# RAG_time

Part 6's advisor got real information two ways: calling a live tool (`imt_taf_list`) or stuffing an entire researched briefing straight into its instructions (Program 14's `briefing_text`, built once for *every* TAF, whether the student ever asks about them or not). That works at ten TAF. It stops working once the source material is a 218-page book, a whole wiki, or a folder of PDFs -- Part 3's context window is finite, and most of that text would be irrelevant to any single question anyway.

**RAG** (Retrieval-Augmented Generation) is the fix: cut the source material into chunks and embed them once, in advance (Part 4's embeddings, applied to passages instead of single tokens), embed the question the same way, and use cosine similarity to pull out only the few chunks actually relevant to it -- then hand *those* to the model, instead of everything.

We'll build it in two stages, because that's how it goes in practice:

1. **On a clean document first** -- the IoT book from Part 2, `documents/PLIDO_BOOK_en.pdf`. One well-structured text, so we can see the machinery work without anything else getting in the way.
2. **Then on the open web** -- an agent that goes and collects real TAF documentation into `documents/`, at which point retrieval quality drops noticeably. Diagnosing *why*, and fixing it, is where most of the real work in a RAG system actually lives.

## From token embeddings to sentence embeddings

Until now, embeddings lived at the token level: Part 3 showed a token is usually just a *piece* of a word, not the whole thing, and Part 4 gave each one its own vector. RAG needs something coarser -- a single vector for an entire sentence, so it can be compared to a whole question. A simple, homemade way to get there with any model: run the text through it, and average ("mean-pool") the resulting per-token vectors into a single one.

> **Prompt for Gemini** (save as `images/sentence_embedding_academic.jpg`):
>
> A clean, minimalist academic diagram. White background, black lines only, sans-serif labels, no color, no photorealism, no shading.
>
> Title at the top: "From Tokens to a Sentence: Mean-Pooling Through a Neural Network"
>
> A left-to-right pipeline with five stages:
> 1. A small box labeled "Sentence" containing the text "The cat sat on the mat."
> 2. An arrow to a box labeled "Tokenizer", from which six small numbered boxes emerge in a row underneath, labeled "tokens".
> 3. An arrow from that row of tokens up into a large box labeled "Neural network (transformer layers)" -- the same network opened up in Part 4.
> 4. Out of that large box, six vertical arrows emerge in a row, each ending in a small vector icon aligned under its own token; beneath this row, a caption: "one hidden-state vector per token -- the token embeddings from Part 4".
> 5. All six vectors converge downward into a single operation box labeled "mean-pool (average)", which points via one final arrow to a single vector icon labeled "sentence embedding -- one fixed-size vector, independent of sentence length".
>
> Below the whole diagram, a caption in the same simple sans-serif style: "A sentence embedding isn't a new kind of network output -- it's the same per-token hidden states the network already produces, just averaged into one vector."

### A quick word on PyTorch and Hugging Face

Two libraries have been doing quiet work since Part 3, without ever being properly introduced -- worth pausing on now that Program 1.1 imports `torch` directly for the first time, rather than just reading tensors a model handed back.

**PyTorch** (`import torch`) is an open-source library for numerical computing and machine learning, originally built by Facebook/Meta AI, and today the dominant framework for training and running deep learning models -- including the LLMs this whole course is built on. Two ideas make it what it is:

* **Tensors**, its core data structure -- essentially a NumPy array that can run on a GPU. Every `.mean()`, `.squeeze()`, and `@` (matrix multiply) used in this part is a tensor operation.
* **Autograd**, automatic differentiation: PyTorch tracks every operation performed on a tensor so it can compute gradients automatically when *training* a model. We're only doing inference here, never training, which is exactly why `sentence_embedding` wraps its work in `torch.no_grad()` below -- without it, PyTorch would keep tracking gradients for a training step that's never going to happen, for nothing.

**Hugging Face's `transformers`** is a separate library, built on top of PyTorch (or TensorFlow, or JAX -- your choice), that packages thousands of pretrained models behind one consistent interface. `AutoTokenizer.from_pretrained(name)` and `AutoModel.from_pretrained(name)`, used since Part 3, work the same way whatever model name you give them -- "Auto" means the library inspects the name and picks the right underlying tokenizer/model class for you.

In [ ]:
# Program 1.1: a homemade sentence embedding, by mean-pooling token vectors

import torch                     # tensors + autograd, see above
import torch.nn.functional as F  # tensor operations that aren't methods on the tensor itself, like cosine_similarity below
from transformers import AutoTokenizer, AutoModel

model_name = "HuggingFaceTB/SmolLM2-135M"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# AutoModel (not AutoModelForCausalLM, Part 3's choice) gives direct access to hidden
# states, without the extra layer that predicts next-token logits -- we don't need that here.
base_model = AutoModel.from_pretrained(model_name)
base_model.eval()

def sentence_embedding(text):
    """A simple sentence embedding: the average of all its tokens' final hidden states."""
    inputs = tokenizer(text, return_tensors="pt")
    with torch.no_grad():
        hidden_states = base_model(**inputs).last_hidden_state  # shape: (1, num_tokens, embedding_dim)
    return hidden_states.mean(dim=1).squeeze(0)  # average over the tokens -> a single vector

sentences = [
    "The cat sat on the mat.",
    "A feline was resting on the rug.",
    "The stock market crashed yesterday.",
]

embeddings = [sentence_embedding(s) for s in sentences]

for i in range(len(sentences)):
    for j in range(i + 1, len(sentences)):
        similarity = F.cosine_similarity(embeddings[i].unsqueeze(0), embeddings[j].unsqueeze(0)).item()
        print(f"{similarity:.3f}  {sentences[i]!r}\n       <->  {sentences[j]!r}")

The two sentences that mean roughly the same thing (the cat/feline ones) score noticeably higher than either has with the unrelated sentence about the stock market -- even though they don't share a single word. That's the whole point of a sentence embedding: it captures meaning, not vocabulary overlap.

## Does this hold across languages?

Within English, the trick works cleanly. But a sentence embedding is only as good as what the underlying model actually learned -- and `SmolLM2-135M` was trained overwhelmingly on English text. Let's translate the cat sentence into French and German, and the stock-market sentence into Spanish, and see whether "meaning over vocabulary" still holds once vocabulary means an entirely different language.

In [ ]:
# Program 1.2: does mean-pooling hold across languages?

sentences = sentences + [
    "le chat est assis sur le tapis.",         # French: same meaning as sentence 0
    "die Katze saß auf der Matte.",            # German: same meaning as sentence 0
    "El mercado de valores se desplomó ayer.", # Spanish: same meaning as sentence 2
]

embeddings = [sentence_embedding(s) for s in sentences]

for i in range(len(sentences)):
    for j in range(i + 1, len(sentences)):
        similarity = F.cosine_similarity(embeddings[i].unsqueeze(0), embeddings[j].unsqueeze(0)).item()
        print(f"{similarity:.3f}  {sentences[i]!r}\n       <->  {sentences[j]!r}")

Look specifically at the German translation: `'The cat sat on the mat.' <-> 'die Katze saß auf der Matte.'` scores *lower* (0.672) than `'The cat sat on the mat.' <-> 'The stock market crashed yesterday.'` (0.746) -- two entirely unrelated English sentences rank as more similar than a sentence and its own faithful German translation.

That's not a bug -- it's a real limit of the technique. `SmolLM2`'s mean-pooling was never trained to align meaning *across* languages; it just averages whatever the model happens to represent, and two English sentences share vocabulary, word order, and token statistics that a simple average leans on heavily, regardless of what they actually mean. French fares a bit better here (Latin script, some shared roots with English), German a bit worse -- but neither is reliable. "Meaning over vocabulary" held within one language; it doesn't automatically survive the trip across languages, and whether it does at all depends entirely on what the underlying model was trained on.

## A real embeddings API

Mean-pooling a small local model's hidden states works, but it's a rough approximation -- `SmolLM2` was never specifically trained to produce good sentence embeddings, in any language. Providers instead offer dedicated **embedding models**, trained precisely for this. Like everywhere else in this course, we can call one through the same OpenAI-compatible client, just changing the endpoint: `embeddings.create` instead of `chat.completions.create`. Let's use Gemini's, on the same sentences.

In [ ]:
# Program 2: sentence embeddings via a real embeddings API (Gemini)

from dotenv import load_dotenv
import os
from openai import OpenAI

load_dotenv(override=True)
google_api_key = os.getenv("GOOGLE_API_KEY")

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)

def gemini_embedding(text):
    response = gemini.embeddings.create(model="gemini-embedding-001", input=text)
    return torch.tensor(response.data[0].embedding)

gemini_embeddings = [gemini_embedding(s) for s in sentences]

for i in range(len(sentences)):
    for j in range(i + 1, len(sentences)):
        similarity = F.cosine_similarity(gemini_embeddings[i].unsqueeze(0), gemini_embeddings[j].unsqueeze(0)).item()
        print(f"{similarity:.3f}  {sentences[i]!r}\n       <->  {sentences[j]!r}")

Same ranking as our homemade version, but with a clearer gap between the related pair and the unrelated one -- exactly what we'd expect from a model actually trained for this task. Look specifically at the German comparison this time: `'The cat sat on the mat.' <-> 'die Katze saß auf der Matte.'` scores 0.856, clearly ahead of `'The cat sat on the mat.' <-> 'The stock market crashed yesterday.'` at 0.612 -- the anomaly from Program 1.2 is gone. A model trained specifically for embeddings doesn't just do better on average; it fixes exactly the kind of failure we just found.

Note also the vector length: `gemini-embedding-001` returns 3072 numbers per sentence, regardless of how long the sentence is -- a fixed-size summary of its meaning, whether it's fed three words or three paragraphs.

### Your turn: does a real embeddings API handle these two better?

You now have two tools that both claim to turn a sentence into a vector: `sentence_embedding` (Program 1, homemade, mean-pooled) and `gemini_embedding` (Program 2, a model actually trained for this). Let's settle it directly, on two languages from entirely different families -- Chinese and Hungarian (not even Indo-European) -- so that if you happen to read one of them, the other still makes the point honestly.

* Embed each sentence in `mystery_sentences` with `sentence_embedding`, and separately with `gemini_embedding`.
* For each, compute its cosine similarity against every sentence in `sentences`, and find the closest match.
* Do the two tools agree, for both sentences? If not, which one gets it right?

Try it yourself before reading on.

In [ ]:
# Program 3: your turn -- test both embedding tools on two unfamiliar languages

mystery_sentences = [
    "股市昨天崩盤了",              # Chinese
    "A tőzsde tegnap összeomlott.", # Hungarian
]

# your code here

With `sentence_embedding` (Program 1), neither mystery sentence lands correctly: for both the Chinese one and the Hungarian one, the closest match is the French cat sentence -- and the true translation, `'The stock market crashed yesterday.'`, ranks near the very bottom of the six both times (dead last for Hungarian). With `gemini_embedding` (Program 2), both mystery sentences correctly land on the stock-market pair (the English original, or its Spanish twin -- both mean the same thing), by the same wide margin you already saw with German.

Same pattern as before, just more dramatic, and now confirmed on two languages that share almost nothing with English or with each other: `SmolLM2-135M` was trained mostly on English and European-language text, so mean-pooling its hidden states works reasonably within a language family it knows, and breaks down entirely somewhere it barely saw any data -- Chinese and Hungarian, in this case, from two completely unrelated families. `gemini-embedding-001` was trained specifically to place same-meaning sentences close together, in any language, and does exactly that here too.

Keep this in mind for Program 4: does a smaller, locally-run model -- built for the same purpose -- do just as well?

## A local model, actually trained for embeddings

There's a practical problem with using Gemini for everything that follows. Three sentences cost three API calls; a 218-page book cut into passages costs **several hundred**, and you re-pay them every time you rebuild the index. Gemini's free tier is generous but not unlimited (Part 2 already ran into its per-day request cap), and burning it on bulk indexing would be a poor trade.

So we'll split the work the way a real system does: a hosted API is fine for occasional, high-value calls, but bulk embedding of a corpus belongs on a model you run yourself. Program 1 already mean-pooled a local model -- the only thing wrong with it was that `SmolLM2` was never *trained* to produce sentence embeddings. Let's keep the exact same mean-pooling code, and swap in a model that was: `multilingual-e5-small` (118M parameters, about the size of the `SmolLM2-135M` we've been using since Part 3 -- so this isn't about a bigger model, it's about a differently *trained* one).

Two details this model expects, which are worth knowing because most embedding models have some equivalent:

* **Prefixes**: stored passages must be prefixed with `passage: ` and questions with `query: `. It was trained that way, and skipping it measurably degrades results.
* **Normalisation**: vectors are scaled to length 1, which makes cosine similarity a plain dot product -- so retrieval later is one matrix multiplication instead of a Python loop.

It's multilingual, which matters here: our documents are in French, our questions often in English -- and, as the exercise just showed, Chinese and Hungarian work too, when the model was actually trained for it.

In [ ]:
# Program 4: a local embedding model, trained for the job -- same three sentences, no API calls

embed_name = "intfloat/multilingual-e5-small"
embed_tokenizer = AutoTokenizer.from_pretrained(embed_name)
embed_model = AutoModel.from_pretrained(embed_name)
embed_model.eval()

def embed(texts, batch_size=32):
    """Embed a list of texts. Same mean-pooling as Program 1, plus length-1 normalisation,
    and batched so a few hundred passages don't take a few hundred forward passes."""
    vectors = []
    for start in range(0, len(texts), batch_size):
        batch = embed_tokenizer(texts[start:start + batch_size], return_tensors="pt",
                                truncation=True, max_length=512, padding=True)
        with torch.no_grad():
            hidden_states = embed_model(**batch).last_hidden_state
        # Mean-pool over real tokens only -- padding tokens must not count towards the average.
        mask = batch["attention_mask"].unsqueeze(-1).float()
        pooled = (hidden_states * mask).sum(dim=1) / mask.sum(dim=1)
        vectors.append(F.normalize(pooled, dim=-1))
    return torch.cat(vectors)

def embed_passages(texts):
    return embed([f"passage: {t}" for t in texts])

def embed_query(text):
    return embed([f"query: {text}"])[0]

local_embeddings = embed_passages(sentences)

for i in range(len(sentences)):
    for j in range(i + 1, len(sentences)):
        # Vectors are normalised, so the dot product IS the cosine similarity.
        similarity = (local_embeddings[i] @ local_embeddings[j]).item()
        print(f"{similarity:.3f}  {sentences[i]!r}\n       <->  {sentences[j]!r}")

# And the mystery sentences, one more time -- still 118M parameters, but trained for exactly this.
for mystery in mystery_sentences:
    mystery_embedding = embed_query(mystery)
    scores = local_embeddings @ mystery_embedding
    print(f"\n{mystery!r} is closest to:")
    print(f"  {sentences[scores.argmax().item()]!r}")

Same ranking again, and both mystery sentences land correctly too -- confirming it really was about training, not size: `multilingual-e5-small` (118M) is smaller than `SmolLM2` (135M) and still gets both right, because it was actually trained on Chinese and Hungarian among dozens of other languages.

From here on, the corpus gets embedded locally, and Gemini stays where it earns its keep -- occasional calls, not bulk indexing.

## Chunking: why we can't embed a whole book

Now the actual corpus: `documents/PLIDO_BOOK_en.pdf`, the IoT book Part 2 stuffed *whole* into the agent's instructions. It's 218 pages, roughly 75,000 words.

We can't embed it as one vector. Partly because a single vector summarising 75,000 words would be so generic it'd match every question equally, but mostly for a harder reason: the model has a **512-token input limit**, so anything past roughly the first 350 words is silently discarded. Not an error, not a warning -- just truncated, and you'd never know from the output.

So we **chunk**: cut the text into passages small enough to embed intact, each one specific enough to be a meaningful answer on its own. Two parameters matter:

* **Size** -- 180 words here, comfortably inside the 512-token limit even with long technical words. Too big and you hit the truncation trap; too small and a passage loses the context that makes it meaningful.
* **Overlap** -- 40 words repeated between consecutive chunks, so a sentence that happens to straddle a boundary still appears whole in one of them.

One more thing this particular PDF forces on us. A book has a table of contents and an index -- pages of `LoRaWAN . . . . . . . . 27, 59`. Those chunks are almost pure punctuation, they carry no meaning, and they pollute results. So we drop any chunk that isn't mostly letters. Real corpora always need some cleanup like this; the only question is which kind.

In [ ]:
# Program 5: chunk the IoT book, drop the junk, and index it -- the whole thing, locally

import time
from pypdf import PdfReader

DOCS_DIR = os.path.abspath(os.path.join(os.getcwd(), "documents"))

book = PdfReader(os.path.join(DOCS_DIR, "PLIDO_BOOK_en.pdf"))
book_text = "\n".join((page.extract_text() or "") for page in book.pages)
print(f"Book: {len(book.pages)} pages, {len(book_text.split()):,} words")

def chunk_text(text, size=180, overlap=40):
    """Cut text into overlapping passages of `size` words."""
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        chunks.append(" ".join(words[start:start + size]))
        start += size - overlap
    return chunks

def is_useful(chunk):
    """Drop table-of-contents and index chunks: mostly dots and page numbers, few real words."""
    letters = sum(character.isalpha() for character in chunk)
    return letters / max(len(chunk), 1) > 0.6

raw_chunks = chunk_text(book_text)
book_chunks = [c for c in raw_chunks if is_useful(c)]
print(f"{len(raw_chunks)} chunks -> {len(book_chunks)} kept "
      f"({len(raw_chunks) - len(book_chunks)} dropped as table-of-contents/index)")

start_time = time.time()
book_vectors = embed_passages(book_chunks)
print(f"Indexed in {time.time() - start_time:.0f}s, entirely on this machine -- zero API calls.")

## Retrieving from the index

The index is built. Retrieval is now the cosine similarity from Part 4, applied at scale: embed the question, compare it to every stored chunk, keep the closest few.

Because the vectors are normalised, that whole comparison is a single matrix multiplication -- `vectors @ question`, one dot product per chunk, done in one call rather than a Python loop over hundreds of entries. This is the same operation a real vector database (FAISS, Chroma, pgvector...) optimises for millions of chunks; at our scale, plain PyTorch is entirely enough.

Note we ask for the top **three** chunks, not just the best one. Retrieval isn't perfect, and giving the model a few candidates lets it pick out the relevant part itself -- a cheap and very effective safety margin.

In [ ]:
# Program 6: retrieve the passages closest to a question

def retrieve(question, chunks, vectors, top_k=3):
    question_vector = embed_query(question)
    scores = vectors @ question_vector          # one dot product per chunk, in one operation
    best = scores.topk(top_k)
    return [(scores[i].item(), chunks[i]) for i in best.indices.tolist()]

for question in ["How does 6LoWPAN compress IPv6 headers?",
                 "Qu'est-ce que le protocole MQTT ?"]:
    print(f"Q: {question}")
    for score, chunk in retrieve(question, book_chunks, book_vectors):
        print(f"  {score:.3f}  {' '.join(chunk.split())[:150]}...")
    print()

Both questions land on the right passage, and the second one is worth a second look: the question is in French, the book is in English, and they share almost no vocabulary -- yet the MQTT passage comes back. That's the multilingual embedding doing exactly what Program 1.2 found `SmolLM2` couldn't: Program 4's `multilingual-e5-small` was actually trained for this, now proving it across 400 passages of a real book instead of six toy sentences.

## Wrapping retrieval as a tool

Like every other capability in this course, retrieval becomes useful to an agent once it's wrapped as a tool. This one doesn't fetch a web page or write a file -- it searches the index we just built, and hands back raw passages for the model to answer from.

Compare this with Part 2's `book_agent`, which put the *entire* book into its instructions on every single call. Same book, same questions, but now the model only ever sees the three passages that matter.

In [ ]:
# Program 7: a RAG agent over the book -- it only ever sees the passages it retrieves

from IPython.display import Markdown, display
from agents import Agent, Runner, function_tool, OpenAIChatCompletionsModel
from openai import AsyncOpenAI

RENNES_BASE_URL = "https://ragarenn.eskemm-numerique.fr/sso/instance@imt/api/"
rennes_client = AsyncOpenAI(base_url=RENNES_BASE_URL, api_key=os.environ["RENNES_API_KEY"])
rennes_model = OpenAIChatCompletionsModel(model="ilaas/mistral-small-4-119b", openai_client=rennes_client)

@function_tool
def search_book(question: str):
    """Search the Internet of Things book for the passages most relevant to a question."""
    passages = retrieve(question, book_chunks, book_vectors)
    return "\n\n---\n\n".join(chunk for _, chunk in passages)

book_rag_agent = Agent(
    name="PLIDO Book RAG Agent",
    instructions="Answer questions about the Internet of Things using the search_book tool. "
                 "Base your answer only on the passages it returns -- if they don't contain "
                 "the answer, say so rather than inventing one.",
    model=rennes_model,
    tools=[search_book],
)

result = await Runner.run(book_rag_agent, "What is 6LoWPAN and why is it needed?", max_turns=6)
display(Markdown(result.final_output))

A correct, grounded answer, built from three passages instead of 75,000 words. That's RAG working exactly as advertised.

It worked this smoothly for a reason worth naming: the book is a *clean* corpus. One document, written by one author, no navigation menus, no duplicated pages, consistent structure throughout. Now let's see what happens when the corpus comes from the open web instead.

## Building a real corpus: the librarian agent

We want documentation about the TAF programs -- and unlike the book, nobody hands us a single tidy PDF. It's scattered across the school's website, brochures, and PDF catalogues.

So we'll do what Part 6 taught us to do: build an agent for it. This one searches with Brave, downloads whatever it finds, extracts plain text (via `pypdf` for PDFs, `BeautifulSoup` for web pages), and saves the result into `documents/`. Two tools, and it decides for itself what's worth keeping.

Note the failure handling in `save_document`. Fetching real documents from the real web fails constantly -- dead links, login walls, pages that turn out to be nearly empty. Rather than crash the agent, every failure returns a *message* the model can read and act on, so it just tries the next result. An agent that can't cope with failing tools is useless outside a demo.

You'll also need your Brave API key from Part 6 (`BRAVE_API_KEY` in `.env`); the free tier covers this comfortably.

In [ ]:
# Program 8: an agent that collects real TAF documentation from the web into documents/

import io
import re
import requests
from bs4 import BeautifulSoup

HEADERS = {"User-Agent": "Mozilla/5.0 (PLIDOagent-course/1.0; educational use)"}
BRAVE_API_KEY = os.environ["BRAVE_API_KEY"]

@function_tool
def search_documents(query: str):
    """Search the web with Brave for documents about IMT Atlantique TAF programs.
    Returns up to 5 result titles and URLs."""
    response = requests.get("https://api.search.brave.com/res/v1/web/search",
                            headers={"Accept": "application/json", "X-Subscription-Token": BRAVE_API_KEY},
                            params={"q": query, "count": 5}, timeout=20)
    time.sleep(1.2)  # the free tier allows about one request per second
    if response.status_code != 200:
        return f"Search failed (HTTP {response.status_code}). Try again or rephrase the query."
    results = response.json().get("web", {}).get("results", [])
    return "\n".join(f"{r['title']}: {r['url']}" for r in results) or "No results found."

def extract_text(response):
    """Plain text out of a PDF or a web page.

    For web pages, dropping the navigation matters more than it looks: menus, footers and
    campus lists are repeated on every page of a site, they mention a little of everything,
    and so they end up moderately close to *every* question -- crowding out the passages
    that are genuinely about one topic. So we strip those tags and keep the main content.
    """
    if "pdf" in response.headers.get("content-type", "").lower():
        pdf = PdfReader(io.BytesIO(response.content))
        return "\n".join((page.extract_text() or "") for page in pdf.pages)

    soup = BeautifulSoup(response.text, "html.parser")
    for tag in soup(["script", "style", "nav", "header", "footer", "aside", "form"]):
        tag.decompose()
    main = soup.find("main") or soup.find("article") or soup.find(attrs={"role": "main"}) or soup
    return main.get_text("\n", strip=True)

@function_tool
def save_document(url: str, filename: str):
    """Download a document (PDF or web page) and save its plain text into documents/<filename>.txt.
    Use a short descriptive filename, without extension. Returns how many characters were saved,
    or an explanation if the document could not be fetched or held too little text."""
    if not re.fullmatch(r"[A-Za-z0-9_-]+", filename):
        return "Invalid filename: use only letters, digits, dashes and underscores."
    try:
        response = requests.get(url, headers=HEADERS, timeout=30)
    except Exception as error:
        return f"Could not fetch {url}: {error}"
    if response.status_code != 200:
        return f"Could not fetch {url}: HTTP {response.status_code}"
    try:
        text = extract_text(response)
    except Exception as error:
        return f"Could not read the document at {url}: {error}"

    text = re.sub(r"\n{3,}", "\n\n", text).strip()
    if len(text) < 1500:
        # Landing pages that only host a download link end up here, and that is the point:
        # they are almost pure title and file metadata, but they mention the topic often
        # enough to outrank genuine documentation if we let them into the corpus.
        return f"Only {len(text)} characters of real content at {url} -- too thin, skipping."

    with open(os.path.join(DOCS_DIR, f"{filename}.txt"), "w") as f:
        f.write(f"SOURCE: {url}\n\n{text}")
    return f"Saved {len(text)} characters to documents/{filename}.txt"

librarian_instructions = """You build a local documentation library about the TAF (Thematique
d'Approfondissement) programs at IMT Atlantique.

Work autonomously -- never ask the user anything. Then:
1. Use search_documents to find pages and PDFs describing IMT Atlantique's TAF programs: their
   content, their syllabus, and the careers they lead to.
2. For each promising result, call save_document with a short descriptive filename. PDF
   catalogues covering several TAF at once are the most valuable of all.
3. Many results will fail -- dead links, login walls, pages with almost no real content. That
   is expected: just move on to the next result, or try a different search.
4. Stop once you have saved 3 useful documents, and reply with a one-line summary.
"""

librarian = Agent(name="TAF Librarian", instructions=librarian_instructions,
                  model=rennes_model, tools=[search_documents, save_document])

result = await Runner.run(librarian, "Build the TAF documentation library.", max_turns=30)
print(result.final_output)

print("\nFiles now in documents/:")
for filename in sorted(f for f in os.listdir(DOCS_DIR) if f.endswith(".txt")):
    size_kb = os.path.getsize(os.path.join(DOCS_DIR, filename)) / 1024
    print(f"  {filename}  ({size_kb:.0f} KB)")

Open `documents/` and look at what came back. In our run the agent saved the official *TAF 2019* catalogue (27 pages, with a real per-programme breakdown), the engineering-degree brochure, and a couple of web pages -- while several other candidates failed with 404s or login walls, exactly as the instructions anticipated.

Two practical notes:

* **Results vary between runs.** The web changes, and the agent picks different sources each time. `documents/backup/` in this repository holds a snapshot of what we collected, so you have a working corpus even if today's searches come back empty.
* **You can add your own.** Anything you drop into `documents/` as a `.txt` file -- a syllabus, your own course notes -- gets indexed by the next program, with no code change. That's the point of keeping the corpus a plain directory of text files.

## The same code, a much worse result

Now let's index this new corpus with *exactly* the code from Program 5 -- same chunking, same junk filter, same embedding model -- and ask it the kind of question a student would actually ask.

In [ ]:
# Program 9.1: index the collected documents with the very same code -- and watch it struggle

def load_documents(directory, chunker):
    """Read every .txt file in a directory, remembering which file each chunk came from."""
    chunks, sources = [], []
    for filename in sorted(f for f in os.listdir(directory) if f.endswith(".txt")):
        text = open(os.path.join(directory, filename)).read()
        for chunk in chunker(text):
            if is_useful(chunk):
                chunks.append(chunk)
                sources.append(filename)
    return chunks, sources

# Fall back on the committed snapshot if the librarian came home empty-handed.
corpus_dir = DOCS_DIR
if not any(f.endswith(".txt") for f in os.listdir(DOCS_DIR)):
    corpus_dir = os.path.join(DOCS_DIR, "backup")
    print("No documents collected -- falling back on documents/backup/\n")

taf_chunks, taf_sources = load_documents(corpus_dir, chunk_text)
taf_vectors = embed_passages(taf_chunks)
print(f"{len(taf_chunks)} chunks from {len(set(taf_sources))} documents\n")

for question in ["Which TAF is about data science and artificial intelligence?",
                 "I want to work on connected devices and industry 4.0"]:
    print(f"Q: {question}")
    for score, chunk in retrieve(question, taf_chunks, taf_vectors):
        print(f"  {score:.3f}  {' '.join(chunk.split())[:110]}...")
    print()

Compare that with the book. There, each question landed on the passage that answered it. Here the top hits are vague -- a page header, a run of programme acronyms, a chunk about medical imaging when you asked about connected devices. The catalogue really does contain a `DASCI – DATA SCIENCE` record and an `IOT – INTERNET OF THINGS` one, and neither comes back.

Nothing is broken: same model, same code, same junk filter. What changed is the **shape of the documents**.

The book is continuous prose. Cut it anywhere and you still get a passage about one topic, because that's how prose works -- a paragraph about 6LoWPAN is surrounded by more text about 6LoWPAN.

A course catalogue is the opposite: a **list of short, self-contained records**, one per programme, each only a dozen lines long. Cutting every 180 words pays no attention to those boundaries, so one chunk ends up holding the tail of one TAF, all of the next, and the start of a third. Its embedding is the average of three unrelated programmes -- close to nothing in particular -- while a question about exactly one of them has nothing precise to match.

The fix isn't a better model or a bigger chunk. It's to **cut where the document says to cut**: the catalogue marks every record with a heading of its own,

```
CYBER – CYBERSECURITY (4+3)
DASCI – DATA SCIENCE: FROM DATA TO DECISION-MAKER (3+4)
IOT – INTERNET OF THINGS FOR THE INDUSTRY 4.0 (3+3)
```

so we split on those instead, and each chunk becomes exactly one programme. Fixed-size chunking stays the fallback for documents with no such structure -- like the book.

In [ ]:
# Program 9.2: cut on the document's own section headings instead of every 180 words

# A TAF record always starts with a heading like "CYBER – CYBERSECURITY (4+3)":
# an acronym, a dash, a title, and the number of semesters.
SECTION_HEADING = re.compile(r"^\s*[A-Z][A-Z0-9&*\-' ]{1,40}\s*[–-]\s+.{3,70}\(\d\+\d\)\s*$")

def chunk_structured(text):
    """Split on section headings when the document has them, fixed-size otherwise."""
    lines = text.splitlines()
    headings = [i for i, line in enumerate(lines) if SECTION_HEADING.match(line)]

    if len(headings) < 3:            # no real structure -- the book takes this path
        return chunk_text(text)

    chunks = []
    if headings[0] > 0:              # whatever comes before the first heading
        chunks += chunk_text("\n".join(lines[:headings[0]]))
    for start, end in zip(headings, headings[1:] + [len(lines)]):
        section = "\n".join(lines[start:end]).strip()
        # One record per chunk -- unless a record is itself too long to embed intact.
        chunks += chunk_text(section) if len(section.split()) > 180 else [section]
    return chunks

taf_chunks, taf_sources = load_documents(corpus_dir, chunk_structured)
taf_vectors = embed_passages(taf_chunks)
print(f"{len(taf_chunks)} chunks, one per programme wherever the document allowed it\n")

for question in ["Which TAF is about data science and artificial intelligence?",
                 "I want to work on connected devices and industry 4.0"]:
    print(f"Q: {question}")
    for score, chunk in retrieve(question, taf_chunks, taf_vectors):
        print(f"  {score:.3f}  {' '.join(chunk.split())[:110]}...")
    print()

Both questions now land on the right record -- `DS`/`DASCI – DATA SCIENCE: FROM DATA TO DECISION-MAKER` and `IOT – INTERNET OF THINGS FOR THE INDUSTRY 4.0` -- where Program 9.1 surfaced neither. Same corpus, same embedding model, same retrieval code. **The only thing that changed is where we cut the text**, and it was worth more than any model upgrade would have been. (While writing this part we tried a bigger embedding model: it made no difference. Chunking did.)

Two caveats worth keeping in mind, because they're the normal condition of RAG rather than defects of this example:

* **Your results will differ from ours.** The librarian collects whatever the web offers today, so your corpus isn't ours. A question whose TAF happens to be missing from the documents you collected will still come back with something vaguely related -- retrieval always returns its closest matches, even when nothing is genuinely close. Being unable to say "I don't know" is a real limitation of plain similarity search.
* **The heading pattern is specific to these catalogues.** `SECTION_HEADING` matches how IMT Atlantique formats a TAF record; another corpus needs another rule -- Markdown `##` headings, numbered clauses, `<h2>` tags. There's no universal chunker, which is exactly why chunking deserves this much attention.

Wire `taf_chunks` into a `@function_tool` exactly like `search_book` in Program 7, and you have Part 6's TAF advisor again -- except its knowledge now comes from real documents you can open and check, rather than descriptions written by hand.

## Key takeaways

* A **sentence embedding** extends Part 4's token embeddings to whole passages -- by mean-pooling a model's hidden states, or via a dedicated embeddings API -- giving a fixed-size vector regardless of length.
* **RAG** means embedding the source material once, in advance, then using cosine similarity to retrieve only the passages relevant to a question -- instead of stuffing everything into every prompt, which is what Part 2's `book_agent` and Part 6's Program 14 both did.
* Retrieval is a third kind of decision-maker alongside Part 6's algorithmic and agentic ones: relevance decided by nearest-neighbour search over embeddings, not by fixed Python branches and not by an LLM reading everything.
* **Bulk embedding belongs on a local model.** A hosted API is right for occasional calls, but indexing a corpus means hundreds of them, re-paid on every rebuild -- enough to exhaust a free tier for no benefit.
* **Every embedding model has a hard input limit** (512 tokens here). Exceed it and your text is silently truncated -- no error, no warning, just half your chunk quietly discarded.
* **Chunking is where RAG quality is won or lost.** The same code that worked on continuous prose failed on a catalogue of short records, because fixed-size cuts ignored the boundaries between them. Cutting on the document's own headings fixed it outright.
* **Extraction quality matters just as much.** A scraped web page is mostly navigation; menus repeated across every page are generic enough to rank against every question. Strip them at extraction time, and thin landing pages fall below the length threshold on their own.
* **Retrieval can't say "I don't know".** It always returns its closest matches, however far away they are -- so a corpus missing the answer produces confident-looking passages about something else.
* When retrieval disappoints, look at your **documents** before reaching for a bigger model. Ours was a chunking problem and an extraction problem -- neither of which a better embedding model would have solved.

## Function reference

A quick reference for the less obvious functions and methods used in this notebook (skipping ones already familiar from earlier parts, like `AutoTokenizer.from_pretrained`, `Agent`, `Runner.run`, and `@function_tool`):

| Function (module) | Arguments | Returns | Used in |
|---|---|---|---|
| `AutoModel.from_pretrained(name)` (`transformers`) | a model name | a model exposing raw hidden states, no next-token head | Programs 1.1, 4 |
| `model(**inputs).last_hidden_state` (`transformers`) | tokenized input | one vector per input token | Programs 1.1, 4 |
| `F.normalize(t, dim=-1)` (`torch.nn.functional`) | a tensor | the same vectors scaled to length 1, so a dot product is a cosine | Program 4 |
| `gemini.embeddings.create(model=, input=)` (`openai`) | a model name, a string | an embedding response (`.data[0].embedding`) | Program 2 |
| `vectors @ query` (`torch`) | a matrix of chunk vectors, one query vector | one similarity score per chunk, in a single operation | Programs 6, 9.1, 9.2 |
| `tensor.topk(k)` (`torch`) | how many results to keep | the k highest scores and their indices | Programs 6, 9.1, 9.2 |
| `PdfReader(path_or_bytes)` (`pypdf`) | a file path, or a `BytesIO` of downloaded bytes | a PDF whose `.pages` expose `.extract_text()` | Programs 5, 8 |
| `soup.find("main")` (`bs4`) | a tag name | the first matching element, or `None` -- used to skip navigation | Program 8 |